# 08 - GEOM-Drugs structural fingerprint guidance

Paper items: **Table 3**, the GEOM-Drugs row of **Table 21**, the GEOM-Drugs columns of
**Tables 19 and 20**, and the inference settings of **Appendix F.8**.

With `RERUN = False` every table below is rebuilt on CPU from `results/geom/`
(provenance: `results/geom/PROVENANCE.md`) and shown next to the number printed in the
paper (`results/geom/paper_values.json`). With `RERUN = True` the last section reruns the
ported steering code at small *n* on one GPU.

The task: sample a target molecule from the GEOM-Drugs val split, take its 1024-bit
OpenBabel FP2 fingerprint `c`, and steer the base flow model with the tilt
`R(G) = exp(-||E(G) - c||^2)`, where `E` is the trained fingerprint head. The reported
metric counts fingerprints directly on the generated molecule, without the guide.

In [1]:
RERUN = False
from _common import *  # noqa: F401,F403

import multimfm.geom  # noqa: F401  (import check: the ported GEOM package)

paper = load_json("geom/paper_values.json")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

## Table 3 - what was actually run

Two runs of the *same* configuration exist, and they disagree. Both are shown; which one
the paper should report is a decision for the authors, not for this notebook.

* **n=64, one seed, four-point (λ, μ) grid** - Slurm job 35670937 (2026-07-28). The
  printed 0.227 / 87.5% is the `λ=0.02, μ=0.5` point of this grid.
* **n=4,096 × 3 seeds** - Slurm job 47833756 (2026-09-22), the same configuration as the
  selected point, sharded 4 × 1,024 per replicate. This is the sample size the paper's
  text claims (App. F.9: "The GEOM-Drugs evaluation generated 4,096 molecules").

In [2]:
grid = load_csv("geom/table3_n64_grid.csv")
grid_view = grid[["reward_lambda", "mu", "num_mols", "unguided_tanimoto", "guided_tanimoto",
                  "tanimoto_gain", "unguided_pb_valid", "guided_pb_valid"]].copy()
grid_view.insert(0, "selected", ["<-- Table 3" if (l == 0.02 and m == 0.5) else ""
                                for l, m in zip(grid.reward_lambda, grid.mu)])
print("n=64, seed 0, job 35670937 (four (lambda, mu) points):")
grid_view.round(4)

n=64, seed 0, job 35670937 (four (lambda, mu) points):


,selected,reward_lambda,mu,num_mols,unguided_tanimoto,guided_tanimoto,tanimoto_gain,unguided_pb_valid,guided_pb_valid
0,<-- Table 3,0.02,0.50,64,0.2072,0.2272,0.0199,0.875,0.8750
1,,0.05,0.25,64,0.2072,0.2175,0.0103,0.875,0.7969
2,,0.05,0.50,64,0.2072,0.1985,-0.0087,0.875,0.7812
3,,0.10,0.50,64,0.2072,0.2090,0.0018,0.875,0.7812


The four grid points span gains of -0.0087 to +0.0199; the reported point is the best of
the four. At n=64 the per-molecule Tanimoto spread makes a single point's standard error
roughly as large as the gain itself, which the paired statistics below quantify.

In [3]:
# Paired per-molecule statistics, recomputed from the per-molecule arrays.
per_mol = load_csv("geom/table3_per_molecule.csv")


def paired_stats(df):
    """Mean +- SEM of (guided - unguided) over molecules whose FP2 is valid in both arms."""
    both = df[(df.unguided_fp_valid == 1) & (df.guided_fp_valid == 1)]
    d = (both.guided_tanimoto - both.unguided_tanimoto).to_numpy()
    sem = float(d.std(ddof=1) / np.sqrt(len(d))) if len(d) > 1 else float("nan")
    return pd.Series({
        "n_molecules": len(df),
        "n_paired_valid": len(d),
        "unguided_tanimoto": df.unguided_tanimoto.mean(),
        "guided_tanimoto": df.guided_tanimoto.mean(),
        "paired_gain": float(d.mean()),
        "paired_gain_sem": sem,
        "gain_in_sem": float(d.mean() / sem) if sem and not np.isnan(sem) else float("nan"),
    })


sel = per_mol[(per_mol.run == "n64_grid") & (per_mol.reward_lambda == 0.02) & (per_mol.mu == 0.5)]
n4096 = per_mol[per_mol.run == "n4096_3seed"]

summary = pd.DataFrame({
    "Table 3 point (n=64, seed 0)": paired_stats(sel),
    "rerun (n=4,096 x 3 seeds)": paired_stats(n4096),
}).T
summary.round(4)

,n_molecules,n_paired_valid,unguided_tanimoto,guided_tanimoto,paired_gain,paired_gain_sem,gain_in_sem
"Table 3 point (n=64, seed 0)",64.0,64.0,0.2072,0.2272,0.0199,0.0104,1.9194
"rerun (n=4,096 x 3 seeds)",11264.0,11260.0,0.2287,0.2241,-0.0046,0.0008,-5.6745


The n=64 point's gain is +0.0199 with a paired SEM of the same order, i.e. it is not
separated from zero; the 4,096 × 3 rerun of the identical configuration gives a *negative*
gain several SEM below zero. Per replicate:

In [4]:
reps = load_csv("geom/table3_n4096_replicates.csv")
stored = reps.set_index("replicate")[["n_paired_valid", "unguided_tanimoto", "guided_tanimoto",
                                      "paired_gain", "paired_gain_sem",
                                      "unguided_pb_valid", "guided_pb_valid"]]
recomputed = n4096.groupby("replicate").apply(paired_stats)
recomputed[["n_molecules", "n_paired_valid"]] = recomputed[["n_molecules", "n_paired_valid"]].astype(int)
print("stored per-replicate summary (results/geom/table3_n4096_replicates.csv):")
display(stored.round(4))
print("recomputed from results/geom/table3_per_molecule.csv:")
recomputed.round(4)

stored per-replicate summary (results/geom/table3_n4096_replicates.csv):


,n_paired_valid,unguided_tanimoto,guided_tanimoto,paired_gain,paired_gain_sem,unguided_pb_valid,guided_pb_valid
replicate,,,,,,,
0,4095,0.2296,0.2241,-0.0054,0.0013,0.8853,0.8604
1,4094,0.2280,0.2242,-0.0037,0.0014,0.8735,0.8389
2,3071,0.2286,0.2238,-0.0047,0.0015,0.8825,0.8506
pooled,11260,0.2287,0.2241,-0.0046,0.0008,0.8802,0.8499


recomputed from results/geom/table3_per_molecule.csv:


,n_molecules,n_paired_valid,unguided_tanimoto,guided_tanimoto,paired_gain,paired_gain_sem,gain_in_sem
replicate,,,,,,,
0.0,4096,4095,0.2296,0.2241,-0.0054,0.0013,-4.0603
1.0,4096,4094,0.2280,0.2242,-0.0037,0.0014,-2.7209
2.0,3072,3071,0.2286,0.2238,-0.0047,0.0015,-3.0565


Replicate 2 has 3 shards instead of 4: seed 21 crashed on a 72-atom target. The base
model tops out at 71 heavy atoms while the val fingerprint split reaches 82, and the
original target sampler padded to 71 regardless. `multimfm.geom.fingerprint.sample_targets`
now skips over-long targets (0.020% of the val split) while walking the same seeded
permutation, so every draw that did not crash is unchanged
(`docs/provenance/geom/data.md`).

### Table 3 against the paper

In [5]:
sel_row = grid[(grid.reward_lambda == 0.02) & (grid.mu == 0.5)].iloc[0]
t3 = {r["method"]: r for r in paper["table3"]["rows"]}
pooled = reps[reps.replicate == "pooled"].iloc[0]

table3 = compare([
    {"item": "Unconditional (TFG-Flow 'Upper bound') Tanimoto",
     "paper": t3["Unconditional generation (TFG-Flow “Upper bound”)"]["tanimoto"],
     "ours": None, "note": "reference value from Lin et al. 2025; not recomputed here"},
    {"item": "TFG-Flow Tanimoto",
     "paper": t3["TFG-Flow (Lin et al., 2025)"]["tanimoto"],
     "ours": None, "note": "reference value from Lin et al. 2025; not recomputed here"},
    {"item": "multiMFM Tanimoto",
     "paper": t3["multiMFM"]["tanimoto"],
     "ours": float(sel_row.guided_tanimoto),
     "note": "n=64, seed 0, best of the 4-point grid (job 35670937)"},
    {"item": "multiMFM Tanimoto (4,096 x 3 rerun)",
     "paper": t3["multiMFM"]["tanimoto"],
     "ours": float(pooled.guided_tanimoto),
     "note": "same configuration at the sample size the text claims"},
    {"item": "base-model Tanimoto for the reported run",
     "paper": None, "ours": float(sel_row.unguided_tanimoto),
     "note": "unguided arm of the same run; the paper prints no unguided Tanimoto"},
    {"item": "multiMFM PB validity (%)",
     "paper": paper["table3"]["multimfm_pb_valid_pct"],
     "ours": 100 * float(sel_row.guided_pb_valid), "note": "same n=64 run"},
    {"item": "base PB validity (%)",
     "paper": paper["table3"]["base_pb_valid_pct"],
     "ours": 100 * float(sel_row.unguided_pb_valid),
     "note": "unguided arm of the reported run; see the Table 21 section"},
    {"item": "molecules evaluated (N)",
     "paper": paper["text_claims"]["num_molecules_evaluated"],
     "ours": float(sel_row.num_mols),
     "note": "App. F.9 says 4,096; the reported numbers come from 64"},
])
table3

,item,paper,ours,status,tol,tol_src,note
0,Unconditional (TFG-Flow 'Upper bound') Tanimoto,0.170,NaN,n/a,NaN,,reference value from Lin et al. 2025; not reco...
1,TFG-Flow Tanimoto,0.208,NaN,n/a,NaN,,reference value from Lin et al. 2025; not reco...
2,multiMFM Tanimoto,0.227,0.227162,match,0.0005,printed,"n=64, seed 0, best of the 4-point grid (job 35..."
3,"multiMFM Tanimoto (4,096 x 3 rerun)",0.227,0.224070,MISMATCH,0.0005,printed,same configuration at the sample size the text...
4,base-model Tanimoto for the reported run,None,0.207217,n/a,NaN,,unguided arm of the same run; the paper prints...
5,multiMFM PB validity (%),87.5,87.500000,match,0.0500,printed,same n=64 run
6,base PB validity (%),92.0,87.500000,MISMATCH,0.0500,printed,unguided arm of the reported run; see the Tabl...
7,molecules evaluated (N),4096,64.000000,MISMATCH,0.5000,printed,"App. F.9 says 4,096; the reported numbers come..."


**The two mismatches are the open question about Table 3** (recorded in
`docs/REPRODUCIBILITY_PLAN.md` §7.1):

1. *Sample size.* The printed 0.227 / 87.5% come from 64 molecules and one seed; the text
   says 4,096. The 4,096 × 3 evaluation of the same configuration gives 0.2241 guided
   versus 0.2287 unguided, a paired gain of -0.0046 ± 0.0008.
2. *Base PB validity.* 92.0% matches no measurement of this base model (next section).

The authors' decision (2026-09-29) was to rerun the row exactly as the paper describes
it and report whatever comes out; that rerun is the next section. Nothing in this
notebook is tuned or reselected to close the gap.

### Does steering improve the quantity it is optimising?

The reward is built from the guide's own prediction, so the sharpest diagnostic is the
*head-predicted* Tanimoto rather than the counted FP2 one. If steering cannot raise the
metric it optimises, the shortfall is not about the guide disagreeing with OpenBabel.

In [6]:
hd = load_json("geom/head_tanimoto_direction.json")
big = hd["n4096x3_submitted_config"]
rows = [{"run": f"n=4,096 x 3, replicate {r['replicate']}", "unguided_head": r["unguided"],
         "guided_head": r["guided"], "head_gain": r["head_gain"]}
        for r in big["per_replicate"]]
rows.append({"run": "n=4,096 x 3, pooled", "unguided_head": big["pooled_unguided_head_tanimoto"],
             "guided_head": big["pooled_guided_head_tanimoto"],
             "head_gain": big["pooled_head_gain"]})
n64 = hd["n64_grid_submitted_point"]
rows.append({"run": "n=64 submitted grid point", "unguided_head": n64["unguided_head_tanimoto"],
             "guided_head": n64["guided_head_tanimoto"],
             "head_gain": n64["guided_head_tanimoto"] - n64["unguided_head_tanimoto"]})
display(pd.DataFrame(rows).round(5))
print(f"shards with a negative head gain: {big['shards_with_negative_head_gain']}"
      f" of {big['shards_total']}")
hs = hd["holdout_sweep"]
print(f"held-out sweep ({hs['points']} points): head gain in "
      f"[{hs['head_gain_min']:+.4f}, {hs['head_gain_max']:+.4f}], all negative: {hs['all_negative']}")

,run,unguided_head,guided_head,head_gain
0,"n=4,096 x 3, replicate 0",0.23500,0.23129,-0.00371
1,"n=4,096 x 3, replicate 1",0.23259,0.23031,-0.00228
2,"n=4,096 x 3, replicate 2",0.23376,0.23078,-0.00297
3,"n=4,096 x 3, pooled",0.23378,0.23079,-0.00299
4,n=64 submitted grid point,0.21648,0.22908,0.01261


shards with a negative head gain: 9 of 11
held-out sweep (5 points): head gain in [-0.0119, -0.0053], all negative: True


So at scale the guide's own objective falls too; it rises only at the single n=64 grid point
that was submitted. That rules out "the guide disagrees with the counted fingerprint" as the
explanation and leaves the guidance strength/scale, the Monte-Carlo estimator, or an
implementation defect. Two checks are queued for that:

* a hand audit of the sign convention in the code path the driver uses found **no sign
  inversion** (`docs/provenance/geom/sign_audit.md`), but noted that the paper's coefficient
  `sigma_t^2/2 = (1-t)/t` is implemented as a constant `mu`, which under-guides ~18x at
  t = 0.1 and over-guides ~4.5x at t = 0.9;
* `python -m multimfm.geom.diagnostics grad-sign` measures whether a step *along* the
  estimated gradient raises the guide reward of fresh look-aheads. Its result lands in
  `results/geom/grad_sign.json`.

In [7]:
# the GPU run writes grad_sign.json; the CPU fallback (reduced size) writes grad_sign_cpu.json
gs = next((RESULTS / "geom" / n for n in ("grad_sign.json", "grad_sign_cpu.json")
           if (RESULTS / "geom" / n).exists()), None)
if gs is not None:
    d = load_json(f"geom/{gs.name}")
    print("source:", gs.name, "| n =", d["num_mols"], "molecules, t =", d["t"],
          "| lambda =", d["reward_lambda"])
    print("verdict:", d["verdict"])
    print(f"grad rms: coords {d['grad_coords_rms']:.4g}  atomics {d['grad_atomics_rms']:.4g}")
    display(pd.DataFrame({"step": list(d["guide_reward_by_step"]),
                          "guide_reward": list(d["guide_reward_by_step"].values())}))
else:
    print("PENDING: neither results/geom/grad_sign.json (GPU) nor grad_sign_cpu.json (CPU)")
    print("exists yet.")
    print("Run (one small GPU job):")
    print("  python -m multimfm.geom.diagnostics grad-sign --device cuda --num-mols 8 \\")
    print("      --out results/geom/grad_sign.json")

PENDING: neither results/geom/grad_sign.json (GPU) nor grad_sign_cpu.json (CPU)
exists yet.
Run (one small GPU job):
  python -m multimfm.geom.diagnostics grad-sign --device cuda --num-mols 8 \
      --out results/geom/grad_sign.json


### How fragile is a 64-molecule estimate?

Drawing 64 paired per-molecule gains (with replacement) from the 4,096 × 3 evaluation of the
same configuration shows how often a 64-molecule run looks as good as the submitted one.

In [8]:
frag = load_json("geom/table3_n64_fragility.json")
print(f"population paired gain (11,260 molecules): {frag['pool_mean_gain']:+.4f} "
      f"(sd {frag['pool_sd_gain']:.4f})")
print(f"P(a 64-molecule estimate is positive)          = {100*frag['p_subsample_gain_positive']:.1f}%")
print(f"P(a 64-molecule estimate >= the submitted +0.0199) = "
      f"{100*frag['p_subsample_gain_ge_submitted']:.2f}%")
print(f"P(at least one of 4 grid points reaches it)    = "
      f"{100*frag['p_at_least_one_of_4_grid_points_ge_submitted_if_independent']:.1f}%")
print(f"95% range of a 64-molecule estimate: "
      f"[{frag['subsample_gain_quantiles']['2.5%']:+.4f}, "
      f"{frag['subsample_gain_quantiles']['97.5%']:+.4f}]")
print()
print("caveat:", frag["caveat"])

population paired gain (11,260 molecules): -0.0046 (sd 0.0855)
P(a 64-molecule estimate is positive)          = 33.6%
P(a 64-molecule estimate >= the submitted +0.0199) = 1.06%
P(at least one of 4 grid points reaches it)    = 4.2%
95% range of a 64-molecule estimate: [-0.0258, +0.0163]

caveat: The bootstrap assumes the n=64 run sampled the same population as the 4,096x3 run. It did not exactly: the n=64 run's unguided Tanimoto was 0.2072 against 0.2287 at 4,096, so its 64 targets were an easier draw. Subsampling noise plus selecting the best of four (lambda, mu) points therefore explains part, but probably not all, of the difference; the target draw explains the rest.


## Table 3 rerun exactly as the paper describes it (authors' decision 1, 2026-09-29)

The authors' decision was to rerun the row as §3.2 / App. F.8 / App. F.9 describe it and
report whatever comes out:

* **4,096 molecules** per seed, **3 seeds** (different target draws and noise), because
  TFG-Flow's reference values are "averaged over 3 random seeds" and carry ±;
* **10 flow-map integration steps, 4 posterior samples, 100-step outer ODE** (App. F.8);
* reward `R(G) = exp(-λ‖E(G)-c‖²)`, targets from the GEOM-Drugs val split, molecules up to
  71 heavy atoms; Tanimoto of the **directly counted** FP2 of the generated molecule
  against the target, with no oracle;
* the **unguided arm shares the noise and the targets**, so it gives the base generator's
  Tanimoto *and* the "Unsteered" PB validity of Table 21 on the same 4,096 molecules.

Two things the paper does not state had to be chosen:

* **λ and μ.** They were selected on a **held-out** target pool (512 targets, seed
  20260929) that the evaluation draw excludes by construction
  (`multimfm.geom.fingerprint.holdout_target_indices`); λ=1 - the literal reading of
  `R(G)=exp(-‖E(G)-c‖²)` - is one of the grid points. The rule was fixed before looking at
  the sweep: largest paired Tanimoto gain among points whose guided PB validity is at most
  2 points below their own unguided arm.
* **The guidance schedule.** Algorithm 2 computes the value gradient at *every* outer step
  with no t window; the implementation that produced every recorded number guides every
  2nd step inside t ∈ [0.1, 0.9]. Both are swept (`schedule` column: `recorded` vs
  `every-step`, the latter over t ∈ [0, 0.95], 0.95 being the student's training horizon
  from Table 20) and the selection covers their union.

In [9]:
sel_path = RESULTS / "geom" / "table3_asdescribed_selection.json"
tune_path = RESULTS / "geom" / "table3_asdescribed_tuning.csv"
if tune_path.exists():
    tune = load_csv("geom/table3_asdescribed_tuning.csv")
    display(tune[["reward_lambda", "mu", "num_mols", "unguided_tanimoto", "guided_tanimoto",
                  "paired_gain", "paired_gain_sem", "unguided_pb_valid", "guided_pb_valid",
                  "pb_drop"]].round(4))
    selection = load_json("geom/table3_asdescribed_selection.json")
    print("selection rule:", selection["rule"])
    print("selected:", {k: selection["selected"][k] for k in ("reward_lambda", "mu")},
          "| held-out gain %+.4f" % selection["selected"]["paired_gain"])
    if selection["paper_literal_lambda_1"]:
        print("\nthe paper's literal lambda=1 on the same held-out targets:")
        display(pd.DataFrame(selection["paper_literal_lambda_1"]).round(4))
else:
    print("PENDING: the held-out (lambda, mu) sweep has not been collected yet.")
    print("Run: sbatch scripts/geom/tune_lambda_mu.sbatch, then")
    print("     bash scripts/geom/collect_table3_asdescribed.sh")

,reward_lambda,mu,num_mols,unguided_tanimoto,guided_tanimoto,paired_gain,paired_gain_sem,unguided_pb_valid,guided_pb_valid,pb_drop
0,0.01,0.25,256,0.2336,0.2265,-0.0070,0.0050,0.8828,0.8750,0.0078
1,0.01,0.50,256,0.2336,0.2243,-0.0093,0.0050,0.8828,0.8516,0.0312
2,0.01,1.00,256,0.2336,0.2188,-0.0148,0.0052,0.8828,0.8320,0.0508
3,0.02,0.25,256,0.2336,0.2279,-0.0057,0.0043,0.8828,0.8477,0.0352
4,0.02,0.50,256,0.2336,0.2191,-0.0145,0.0053,0.8828,0.8477,0.0352


selection rule: largest paired Tanimoto gain on the held-out targets among points whose guided PB validity is at most 0.020 below the unguided validity of the same run; if no point qualifies, the largest gain overall
selected: {'reward_lambda': 0.01, 'mu': 0.25} | held-out gain -0.0070


### The as-described evaluation

In [10]:
asdesc_path = RESULTS / "geom" / "table3_asdescribed.json"
if asdesc_path.exists():
    asdesc = load_json("geom/table3_asdescribed.json")
    seeds = load_csv("geom/table3_asdescribed_seeds.csv")
    if not asdesc["complete"]:
        print("PARTIAL: not every shard has finished; numbers below cover what exists.")
    print("config:", json.dumps({k: asdesc["config"][k] for k in
          ("molecules_per_seed", "sample_steps", "mfm_steps", "value_samples",
           "reward_lambda", "mu")}, indent=1))
    display(seeds[["seed_group", "n_molecules", "unguided_tanimoto", "guided_tanimoto",
                   "paired_gain", "paired_gain_sem", "unguided_pb_valid",
                   "guided_pb_valid"]].round(4))
    m, u = asdesc["multimfm"], asdesc["base_generator_unguided"]
    print(f"multiMFM Tanimoto   {m['tanimoto_mean']:.4f} +- {m['tanimoto_se']:.4f}  (SE over seeds)")
    print(f"base (unguided)     {u['tanimoto_mean']:.4f} +- {u['tanimoto_se']:.4f}")
    print(f"paired gain         {asdesc['paired_gain']['mean']:+.4f} +- {asdesc['paired_gain']['se']:.4f}")
    print(f"PB validity         multiMFM {100*m['pb_valid_mean']:.1f}% +- {100*m['pb_valid_se']:.1f}"
          f"   base {100*u['pb_valid_mean']:.1f}% +- {100*u['pb_valid_se']:.1f}")
else:
    asdesc = None
    print("PENDING: the as-described 4,096 x 3 evaluation has not been collected yet.")
    print("Run: LAMBDA=<selected> MU=<selected> sbatch scripts/geom/table3_asdescribed.sbatch,")
    print("     then bash scripts/geom/collect_table3_asdescribed.sh")

PENDING: the as-described 4,096 x 3 evaluation has not been collected yet.
Run: LAMBDA=<selected> MU=<selected> sbatch scripts/geom/table3_asdescribed.sbatch,
     then bash scripts/geom/collect_table3_asdescribed.sh


### All three runs of the multiMFM row side by side

The submitted value, the earlier 4,096 × 3 rerun of that same configuration, and the
as-described rerun. TFG-Flow's two reference values (its own GEOM-Drug rows, verified
against its paper in `results/geom/tfgflow_reference.json`) are shown for scale.

In [11]:
tfg = load_json("geom/tfgflow_reference.json")["source_paper"]["values"]["GEOM-Drug"]
rows = [
    {"run": "TFG-Flow unconditional (their upper bound)", "N": None, "seeds": 3,
     "tanimoto": tfg["upper_bound"], "err": tfg["upper_bound_err"], "pb_valid_pct": None,
     "source": "Lin et al. 2025, Table 3"},
    {"run": "TFG-Flow guided", "N": None, "seeds": 3,
     "tanimoto": tfg["tfg_flow"], "err": tfg["tfg_flow_err"], "pb_valid_pct": None,
     "source": "Lin et al. 2025, Table 3"},
    {"run": "multiMFM as submitted (n=64, 1 seed, best of 4 grid points)", "N": 64, "seeds": 1,
     "tanimoto": float(sel_row.guided_tanimoto), "err": None,
     "pb_valid_pct": 100 * float(sel_row.guided_pb_valid), "source": "job 35670937"},
    {"run": "base generator, same run", "N": 64, "seeds": 1,
     "tanimoto": float(sel_row.unguided_tanimoto), "err": None,
     "pb_valid_pct": 100 * float(sel_row.unguided_pb_valid), "source": "job 35670937"},
    {"run": "multiMFM, same config at 4,096 x 3 (2026-09-22)", "N": 4096, "seeds": 3,
     "tanimoto": float(pooled.guided_tanimoto), "err": None,
     "pb_valid_pct": 100 * float(pooled.guided_pb_valid), "source": "job 47833756"},
    {"run": "base generator, same run", "N": 4096, "seeds": 3,
     "tanimoto": float(pooled.unguided_tanimoto), "err": None,
     "pb_valid_pct": 100 * float(pooled.unguided_pb_valid), "source": "job 47833756"},
]
if asdesc is not None:
    m, u = asdesc["multimfm"], asdesc["base_generator_unguided"]
    n_per_seed = asdesc["config"]["molecules_per_seed"]
    n_seeds = len(asdesc["per_seed"])
    tail = "" if asdesc["complete"] else " [PARTIAL]"
    rows += [
        {"run": "multiMFM as described" + tail, "N": n_per_seed, "seeds": n_seeds,
         "tanimoto": m["tanimoto_mean"], "err": m["tanimoto_se"],
         "pb_valid_pct": 100 * m["pb_valid_mean"], "source": "as-described rerun"},
        {"run": "base generator, same run" + tail, "N": n_per_seed, "seeds": n_seeds,
         "tanimoto": u["tanimoto_mean"], "err": u["tanimoto_se"],
         "pb_valid_pct": 100 * u["pb_valid_mean"], "source": "as-described rerun"},
    ]
else:
    rows.append({"run": "multiMFM as described", "N": 4096, "seeds": 3, "tanimoto": None,
                 "err": None, "pb_valid_pct": None, "source": "PENDING - see above"})
side_by_side = pd.DataFrame(rows)
side_by_side.round(4)

,run,N,seeds,tanimoto,err,pb_valid_pct,source
0,TFG-Flow unconditional (their upper bound),NaN,3,0.1700,0.001,NaN,"Lin et al. 2025, Table 3"
1,TFG-Flow guided,NaN,3,0.2080,0.002,NaN,"Lin et al. 2025, Table 3"
2,"multiMFM as submitted (n=64, 1 seed, best of 4...",64.0,1,0.2272,NaN,87.5000,job 35670937
3,"base generator, same run",64.0,1,0.2072,NaN,87.5000,job 35670937
4,"multiMFM, same config at 4,096 x 3 (2026-09-22)",4096.0,3,0.2241,NaN,84.9876,job 47833756
5,"base generator, same run",4096.0,3,0.2287,NaN,88.0238,job 47833756
6,multiMFM as described,4096.0,3,NaN,NaN,NaN,PENDING - see above


Note the scale: this base generator's *unguided* Tanimoto (≈0.23 at n=4,096) is already
above TFG-Flow's *guided* 0.208, and TFG-Flow's GEOM-Drug setup caps molecules at 30
atoms against 71 here, so the three numbers are not measured on the same molecule-size
distribution. What the rerun tests is the *gain* from steering, which is what multiMFM
claims.

### Other configurations on record (not in the paper)

Context for the row: the same reward steered with teacher look-aheads (GLASS) instead of
the flow map, and the earlier student. None of these is the paper's number.

In [12]:
glass = load_csv("geom/glass_steering_grid_n64.csv")
first = load_csv("geom/n4096_first_run_shards.csv")
older = load_csv("geom/mfm_steering_grid_n64_older_student.csv")
smoke = load_csv("geom/table3_repro_smoke_n64.csv")

context = pd.DataFrame([
    {"run": "GLASS look-ahead, n=64 grid (big teacher)", "points": int((glass.teacher_label == "big").sum()),
     "best_gain": glass[glass.teacher_label == "big"].tanimoto_gain.max(),
     "median_gain": glass[glass.teacher_label == "big"].tanimoto_gain.median()},
    {"run": "GLASS look-ahead, n=64 grid (xlarge teacher)", "points": int((glass.teacher_label == "xlarge").sum()),
     "best_gain": glass[glass.teacher_label == "xlarge"].tanimoto_gain.max(),
     "median_gain": glass[glass.teacher_label == "xlarge"].tanimoto_gain.median()},
    {"run": "multiMFM, n=64 grid, earlier (Stage-2a) student", "points": len(older),
     "best_gain": older.tanimoto_gain.max(), "median_gain": older.tanimoto_gain.median()},
    {"run": "first n=4,096 run (job 35640178), GLASS", "points": int((first.value_sampler == "glass").sum()),
     "best_gain": first[first.value_sampler == "glass"].tanimoto_gain.max(),
     "median_gain": first[first.value_sampler == "glass"].tanimoto_gain.median()},
    {"run": "first n=4,096 run (job 35640178), multiMFM (Stage-2a student)",
     "points": int((first.value_sampler == "mfm").sum()),
     "best_gain": first[first.value_sampler == "mfm"].tanimoto_gain.max(),
     "median_gain": first[first.value_sampler == "mfm"].tanimoto_gain.median()},
    {"run": "post-purge rerun of the Table 3 point (job 47833754)", "points": len(smoke),
     "best_gain": smoke.tanimoto_gain.max(), "median_gain": smoke.tanimoto_gain.median()},
])
context.round(4)

,run,points,best_gain,median_gain
0,"GLASS look-ahead, n=64 grid (big teacher)",12,0.0305,0.0133
1,"GLASS look-ahead, n=64 grid (xlarge teacher)",8,0.0058,-0.0056
2,"multiMFM, n=64 grid, earlier (Stage-2a) student",9,0.0049,-0.0015
3,"first n=4,096 run (job 35640178), GLASS",4,0.0092,0.0058
4,"first n=4,096 run (job 35640178), multiMFM (St...",4,-0.0021,-0.0028
5,post-purge rerun of the Table 3 point (job 478...,1,0.0156,0.0156


The last row is the reproduction check: the exact Table 3 command, same seed and
checkpoints, re-run two months later under the current environment gives a gain of
+0.0156 against the recorded +0.0199 (guided 0.2255 vs 0.2272). Guided sampling is not
bit-reproducible (non-deterministic CUDA backward), so this is the scale of run-to-run
noise at n=64 - itself comparable to the effect being reported.

## Table 21, GEOM-Drugs row - PoseBusters validity

The paper's row is: N = 4,096, unsteered 92.0%, multiMFM 87.5%, multiMFM-SS not run.
What the records contain:

In [13]:
t21 = load_json("geom/table21_geom_row.json")
rec = t21["records"]
curve = load_csv("geom/teacher_training_pb_validity.csv")

pb_table = compare([
    {"item": "multiMFM PB validity (%)", "paper": t21["paper"]["multimfm_pct"],
     "ours": rec["selected_run_n64"]["multimfm_pb_pct"],
     "note": f"job {rec['selected_run_n64']['slurm_job']}, N={rec['selected_run_n64']['N']}"},
    {"item": "multiMFM PB validity (%), 4,096 x 3", "paper": t21["paper"]["multimfm_pct"],
     "ours": rec["rerun_n4096x3"]["multimfm_pb_pct"],
     "note": "same configuration, 11 shards of 1,024"},
    {"item": "unsteered PB validity (%)", "paper": t21["paper"]["unsteered_pct"],
     "ours": rec["selected_run_n64"]["unsteered_pb_pct"],
     "note": "unguided arm of the reported run (same noise and targets)"},
    {"item": "unsteered PB validity (%), 4,096 x 3", "paper": t21["paper"]["unsteered_pct"],
     "ours": rec["rerun_n4096x3"]["unsteered_pb_pct"],
     "note": "unguided arm of the rerun"},
    {"item": "unsteered PB validity (%), selected checkpoint at training validation",
     "paper": t21["paper"]["unsteered_pct"],
     "ours": rec["base_model_training_validation"]["epoch_38_selected_checkpoint_pct"],
     "note": "wandb val/pb_intersection, N=100, no-strain PB config"},
    {"item": "molecules evaluated (N)", "paper": t21["paper"]["N"],
     "ours": float(rec["selected_run_n64"]["N"]),
     "note": "the row's two percentages come from the N=64 run"},
])
pb_table

,item,paper,ours,status,tol,tol_src,note
0,multiMFM PB validity (%),87.5,87.500000,match,0.05,printed,"job 35670937, N=64"
1,"multiMFM PB validity (%), 4,096 x 3",87.5,84.987571,MISMATCH,0.05,printed,"same configuration, 11 shards of 1,024"
2,unsteered PB validity (%),92.0,87.500000,MISMATCH,0.05,printed,unguided arm of the reported run (same noise a...
3,"unsteered PB validity (%), 4,096 x 3",92.0,88.023793,MISMATCH,0.05,printed,unguided arm of the rerun
4,"unsteered PB validity (%), selected checkpoint...",92.0,98.000000,MISMATCH,0.05,printed,"wandb val/pb_intersection, N=100, no-strain PB..."
5,molecules evaluated (N),4096,64.000000,MISMATCH,0.50,printed,the row's two percentages come from the N=64 run


### Where does 92.0% come from?

No surviving record measures this base model's PB validity on 4,096 molecules. `92.0%`
does occur in the teacher's *training-time* validation (100 molecules per epoch, tabasco's
`posebusters_no_strain.yaml`), at three epochs - but not at the epoch of the released
checkpoint, which logged 98.0%:

In [14]:
print("epochs whose training validation PB is exactly 92.0%:",
      rec["base_model_training_validation"]["epochs_with_exactly_92_0_pct"])
print("selected checkpoint (epoch 38):",
      rec["base_model_training_validation"]["epoch_38_selected_checkpoint_pct"], "%")
print("mean over the last 10 epochs:",
      round(rec["base_model_training_validation"]["mean_last_10_epochs_pct"], 2), "%")
curve.tail(12)[["epoch", "global_step", "pb_intersection", "pb_internal_steric_clash",
                "pb_bond_lengths", "pb_bond_angles"]]

epochs whose training validation PB is exactly 92.0%: [26, 28, 37]
selected checkpoint (epoch 38): 98.0 %
mean over the last 10 epochs: 89.9 %


,epoch,global_step,pb_intersection,pb_internal_steric_clash,pb_bond_lengths,pb_bond_angles
27,27,64119,0.91,0.92,1.00,1.00
28,28,66409,0.92,0.93,0.99,0.98
29,29,68699,0.91,0.91,0.96,0.96
30,30,70989,0.86,0.87,0.95,0.96
31,31,73279,0.91,0.92,0.98,0.97
32,32,75569,0.91,0.91,0.98,0.98
33,33,77859,0.84,0.85,0.93,0.93
34,34,80149,0.90,0.92,0.97,0.96
35,35,82439,0.85,0.86,0.93,0.93
36,36,84729,0.91,0.91,0.97,0.97


The standalone evaluations of other checkpoints of the same run
(`results/geom/teacher_pb_curve.csv`, N=200, 200 steps) never covered step 89,310 and peak
at 90.5%. So the honest statements available today are: 87.5% (N=64) or 88.0%
(N=4,096 × 3) for the unsteered base model under the GEOM PB configuration, both measured
in the same runs as the multiMFM number. Establishing a 4,096-molecule base number
requires a new run: `NUM_SAMPLES=4096 sbatch scripts/geom/eval_pb_validity.sbatch`.

In [15]:
# (the first row's "checkpoint" field is verbatim from the source record: that early
# checkpoint was deleted after it was evaluated.)
load_csv("geom/teacher_pb_curve.csv").query("curve == 'big_curve'")[
    ["checkpoint", "epoch", "global_step", "weights", "num_samples", "pb_valid"]]

,checkpoint,epoch,global_step,weights,num_samples,pb_valid
0,(overwritten; evaluated before deletion),1,4580,raw(default),200,0.455
1,epoch=4-step=11450.ckpt,4,11450,raw(default),200,0.770
2,epoch=5-step=13740.ckpt,5,13740,raw(default),200,0.815
3,epoch=6-step=16030.ckpt,6,16030,raw(default),200,0.800
4,epoch=7-step=18320.ckpt,7,18320,raw(default),200,0.840
5,epoch=8-step=20610.ckpt,8,20610,raw(default),200,0.905
6,epoch=9-step=22900.ckpt,9,22900,raw(default),200,0.885
7,epoch=10-step=25190.ckpt,10,25190,raw(default),200,0.815
8,epoch=11-step=27480.ckpt,11,27480,raw(default),200,0.820
9,epoch=23-step=54960.ckpt,23,54960,raw(default),200,0.830


## Table 19, GEOM-Drugs column - base-model training

Rebuilt from the resolved Hydra config of the run that produced the released checkpoint
(`checkpoints/geom/teacher/hydra/config.yaml`) plus the `data_stats` stored inside it.

In [16]:
t19 = load_json("geom/table19_geom_hparams.json")
p19 = paper["table19_geom_column"]

rows = [
    ("Training set (molecules)", p19["training_set"], f"{t19['training_set_molecules_processed_pt']:,} processed "
     f"/ {t19['training_set_molecules_fingerprint_lmdb']:,} fingerprint records"),
    ("Hydrogen atoms", p19["hydrogen_atoms"], t19["hydrogen_atoms"]),
    ("Maximum heavy atoms in data", p19["max_heavy_atoms_in_data"], t19["max_heavy_atoms_in_data"]),
    ("Atom-vector dimension", p19["atom_vector_dimension"], t19["atom_vector_dimension"]),
    ("Coordinate normalization divisor", p19["coordinate_normalization_divisor"],
     t19["coordinate_normalization_divisor"]),
    ("Probability path", p19["probability_path"], t19["probability_path"].capitalize()),
    ("Training-time distribution", p19["training_time_distribution"], t19["training_time_distribution"]),
    ("Hidden width", p19["hidden_width"], t19["hidden_width"]),
    ("Transformer layers", p19["transformer_layers"], t19["transformer_layers"]),
    ("Attention heads", p19["attention_heads"], t19["attention_heads"]),
    ("Cross-attention", p19["cross_attention"], "Enabled" if t19["cross_attention"] else "Disabled"),
    ("Optimizer", p19["optimizer"], t19["optimizer"]),
    ("Optimizer (beta1, beta2)", tuple(p19["optimizer_betas"]), tuple(t19["optimizer_betas"])),
    ("Learning rate", p19["learning_rate"], t19["learning_rate"]),
    ("Learning-rate schedule", p19["learning_rate_schedule"], t19["learning_rate_schedule"].capitalize()),
    ("Weight decay", p19["weight_decay"], t19["weight_decay"]),
    ("Gradient clipping (l2)", p19["gradient_clipping_l2"], t19["gradient_clipping_l2"]),
    ("EMA decay", p19["ema_decay"], t19["ema_decay"]),
    ("Batch before rotation copies", p19["batch_before_rotation_copies"], t19["batch_before_rotation_copies"]),
    ("Additional rotation copies", p19["additional_rotation_copies"], t19["additional_rotation_copies"]),
    ("Batch including rotation copies", p19["batch_including_rotation_copies"],
     t19["batch_including_rotation_copies"]),
    ("Selected checkpoint step", p19["selected_checkpoint_step"], t19["selected_checkpoint_step"]),
    ("Coordinate loss coefficient", p19["coordinate_loss_coefficient"], t19["coordinate_loss_coefficient"]),
    ("Atom loss coefficient", p19["atom_loss_coefficient"], t19["atom_loss_coefficient"]),
]


def agree(paper_value, ours):
    if isinstance(paper_value, (int, float)) and isinstance(ours, (int, float)):
        return "match" if abs(float(paper_value) - float(ours)) < 1e-9 else "MISMATCH"
    return "match" if str(paper_value).strip().lower() == str(ours).strip().lower() else "check"


t19_view = pd.DataFrame(
    [{"hyperparameter": k, "paper": v, "from checkpoint/config": o, "status": agree(v, o)}
     for k, v, o in rows]
)
t19_view

,hyperparameter,paper,from checkpoint/config,status
0,Training set (molecules),~1.17M conformers,"1,141,716 processed / 1,172,182 fingerprint re...",check
1,Hydrogen atoms,Excluded,excluded,match
2,Maximum heavy atoms in data,71,71,match
3,Atom-vector dimension,9,9,match
4,Coordinate normalization divisor,2.0,2.0,match
5,Probability path,Linear,Linear,match
6,Training-time distribution,"Beta(1.8, 1)","Beta(1.8, 1)",match
7,Hidden width,256,256,match
8,Transformer layers,16,16,match
9,Attention heads,8,8,match


Everything numeric agrees. The two `check` rows are prose formatting, not disagreements
(the training-set row shows both counts behind the paper's "~1.17M"; "Linear"/"Enabled"
are spellings of the same setting).

Two points the table does not say: the committed experiment config in the source repo
said `batch_size: 256` while the released run used 512 (the repo's
`configs/geom/experiment/geom_dfm.yaml` is set to 512, the value the run used), and the
model's positional capacity is 90 atoms even though the data - and the samplers - cap at 71.

## Table 20, GEOM-Drugs column - MFM distillation

Rebuilt from the `args` / `student_cfg` dictionaries stored inside the released student
checkpoint. These describe the final phase (Stage 2b) only; the full four-leg lineage is
in `docs/provenance/geom/checkpoints.md`.

In [17]:
t20 = load_json("geom/table20_geom_hparams.json")
p20 = paper["table20_geom_column"]
pairs = [
    ("Hidden width / layers / heads", f"{p20['hidden_width']} / {p20['layers']} / {p20['heads']}",
     f"{t20['hidden_width']} / {t20['layers']} / {t20['heads']}"),
    ("Conditioning", p20["conditioning"], t20["conditioning"].capitalize()),
    ("Velocity parameterization", p20["velocity_parameterization"], t20["velocity_parameterization"].capitalize()),
    ("Batch per GPU", p20["batch_per_gpu"], t20["batch_per_gpu"]),
    ("Number of GPUs", p20["num_gpus"], t20["num_gpus"]),
    ("Global batch", p20["global_batch"], t20["global_batch"]),
    ("Selected updates in phase", p20["selected_updates_in_phase"], t20["selected_updates_in_phase"]),
    ("Cosine schedule horizon", p20["cosine_schedule_horizon"], t20["cosine_schedule_horizon"]),
    ("Optimizer", p20["optimizer"], t20["optimizer"]),
    ("Peak learning rate", p20["peak_learning_rate"], t20["peak_learning_rate"]),
    ("Warmup steps", p20["warmup_steps"], t20["warmup_steps"]),
    ("Gradient clipping (l2)", p20["gradient_clipping_l2"], t20["gradient_clipping_l2"]),
    ("EMA decay", p20["ema_decay"], t20["ema_decay"]),
    ("Objective", p20["objective"], t20["objective"]),
    ("Coordinate / atom weights", f"{p20['coordinate_weight']} / {p20['atom_weight']}",
     f"{t20['coordinate_weight']} / {t20['atom_weight']}"),
    ("Maximum training time", p20["maximum_training_time_t_max"], t20["maximum_training_time_t_max"]),
    ("Conditioning states", p20["conditioning_states"], t20["conditioning_states"]),
    ("Extra mass at t_cond = 0", p20["extra_mass_at_tcond_0"], t20["extra_mass_at_tcond_0"]),
]
pd.DataFrame([{"hyperparameter": k, "paper": v, "from checkpoint": o, "status": agree(v, o)}
              for k, v, o in pairs])

,hyperparameter,paper,from checkpoint,status
0,Hidden width / layers / heads,256 / 16 / 8,256 / 16 / 8,match
1,Conditioning,Additive,Additive,match
2,Velocity parameterization,Endpoint,Endpoint,match
3,Batch per GPU,96,96,match
4,Number of GPUs,4,4,match
5,Global batch,384,384,match
6,Selected updates in phase,10000,10000,match
7,Cosine schedule horizon,30000,30000,match
8,Optimizer,Adam,Adam,match
9,Peak learning rate,0.0001,0.0001,match


### Settings Table 20 omits

The recorded phase used several settings the table does not list, and the distillation set
was a subset of the training data:

In [18]:
om = dict(t20["omitted_from_table20"])
om_view = pd.DataFrame({"setting": list(om), "value in the released run": [om[k] for k in om]})
print("distilled on {:,} of {:,} available molecules ({})".format(
    t20["distillation_data"]["limit_data"], t20["distillation_data"]["available_molecules"],
    t20["distillation_data"]["note"]))
om_view

distilled on 150,000 of 1,141,716 available molecules (the GEOM MFM was distilled on 150k of the ~1.17M molecules)


,setting,value in the released run
0,max_jump,0.15
1,anneal_frac,0.2
2,esd_delta_clip,8.0
3,esd_target_clip,10.0
4,diag_weight,2.0
5,esd_weight,1.0
6,t_cond_power,2.0
7,lr_schedule,cosine


## Appendix F.8 - inference settings used for the GEOM row

"10 flow-map integration steps and four posterior samples within a 100-step outer ODE
trajectory" - read back from the run records:

In [19]:
rec_run = grid[(grid.reward_lambda == 0.02) & (grid.mu == 0.5)].iloc[0]
pd.DataFrame([
    {"setting": "flow-map inner steps", "paper": 10, "recorded": int(rec_run.mfm_steps),
     "where": "App. F.8"},
    {"setting": "posterior samples", "paper": 4, "recorded": int(rec_run.value_samples),
     "where": "App. F.8"},
    {"setting": "outer ODE steps", "paper": 100, "recorded": int(rec_run.sample_steps),
     "where": "App. F.8"},
    {"setting": "maximum heavy atoms", "paper": 71, "recorded": 71, "where": "Sec. 3.2 / Table 19"},
])

,setting,paper,recorded,where
0,flow-map inner steps,10,10,App. F.8
1,posterior samples,4,4,App. F.8
2,outer ODE steps,100,100,App. F.8
3,maximum heavy atoms,71,71,Sec. 3.2 / Table 19


## Verification of the port

The code in `src/multimfm/geom/` was re-run on a GPU at the exact Table 3 point (n=64,
seed 0) and compared with the two earlier runs of the same configuration, and the
released base checkpoint was measured under the GEOM PoseBusters configuration for the
first time (`results/geom/port_verification.json`).

In [20]:
ver = load_json("geom/port_verification.json")
runs = pd.DataFrame(ver["steering_paper_point"]["runs"])
display(runs[["label", "slurm_job", "gpu", "env", "unguided_tanimoto", "guided_tanimoto",
              "gain", "unguided_pb", "guided_pb"]])
print("paired SEM of the gain at n=64:", ver["steering_paper_point"]["paired_gain_sem_at_n64"])
print()
pbv = ver["base_model_pb_validity"]
print(f"base checkpoint PB validity, GEOM PB config, N={pbv['num_samples']} @ {pbv['num_steps']} steps, "
      f"{pbv['weights']} weights: {100 * pbv['pb_valid']:.1f}%  (job {pbv['slurm_job']})")

,label,slurm_job,gpu,env,unguided_tanimoto,guided_tanimoto,gain,unguided_pb,guided_pb
0,original (paper Table 3),35670937,H200,tabasco-gpu,0.2072,0.2272,0.0199,0.875,0.875
1,authors' post-purge rerun,47833754,H200,multimfm,0.2100,0.2255,0.0156,0.875,0.844
2,ported code in this repo,49256831,A100-SXM4-40GB,multimfm,0.2117,0.2159,0.0042,0.875,0.797


paired SEM of the gain at n=64: 0.0104

base checkpoint PB validity, GEOM PB config, N=100 @ 200 steps, ema weights: 88.0%  (job 49256841)


Guided sampling is not bit-reproducible and TF32 matmuls differ between GPU generations,
so the three runs do not agree exactly: the authors' own rerun of the *unchanged* script
already moved the unguided arm from 0.2072 to 0.2100 and the gain from +0.0199 to +0.0156.
The ported code's unguided arm (0.2117) is within 0.005 of both, and its gain (+0.0042) is
1.5 paired SEM from the recorded one - at n=64 neither run separates the port from the
original nor the effect from zero.

The fresh base-model PB measurement (88.0% at N=100 under the Table 3 PB configuration)
agrees with the 87.5% / 88.0% unguided arms and not with Table 21's 92.0%.

## RERUN = True: rerun the ported code at small n on one GPU

Runs `multimfm.geom.mfm_steering` (the code ported into this repo) at the paper's
configuration with n=16 and reports Tanimoto and PoseBusters validity for both arms. n=16
is far too small for the effect to be resolved - the point is that the pipeline runs and
lands in the same range as the stored numbers. Use
`sbatch scripts/geom/steer_table3.sbatch` for the full-size runs.

In [21]:
if RERUN:
    import torch
    from multimfm.geom import steering

    assert torch.cuda.is_available(), "RERUN=True needs a GPU; submit scripts/geom/steer_table3.sbatch"
    out_dir = REPO / "outputs" / "geom" / "notebook_rerun"
    parser = __import__("argparse").ArgumentParser()
    steering.add_common_arguments(parser, value_sampler="mfm")
    args = parser.parse_args([
        "--num-mols", "16", "--sample-steps", "100", "--mfm-steps", "10",
        "--value-samples", "4", "--value-batch-size", "8",
        "--guide-every", "2", "--guide-min-t", "0.1", "--guide-max-t", "0.9",
        "--mu", "0.5", "--reward-lambda", "0.02", "--seed", "0",
        "--device", "cuda", "--tag", "notebook_n16", "--output-dir", str(out_dir),
    ])
    result = steering.run_steering(args, value_sampler="mfm")
    rerun_view = compare([
        {"item": "unguided Tanimoto (n=16)", "paper": float(sel_row.unguided_tanimoto),
         "ours": result["unguided"]["tanimoto_real_mean"],
         "note": "'paper' column = the stored n=64 run, not the paper"},
        {"item": "guided Tanimoto (n=16)", "paper": float(sel_row.guided_tanimoto),
         "ours": result["guided"]["tanimoto_real_mean"],
         "note": "stored n=64 run for reference"},
        {"item": "unguided PB validity (n=16)", "paper": float(sel_row.unguided_pb_valid),
         "ours": result["unguided"]["pb_valid"], "note": "stored n=64 run for reference"},
        {"item": "guided PB validity (n=16)", "paper": float(sel_row.guided_pb_valid),
         "ours": result["guided"]["pb_valid"], "note": "stored n=64 run for reference"},
    ], atol=0.05)
    display(rerun_view)
    print("gain at n=16:", round(result["tanimoto_gain"], 4),
          "| stored gain at n=64:", round(float(sel_row.tanimoto_gain), 4))
else:
    print("RERUN = False: skipping the GPU rerun. Stored value of the same configuration:")
    print(f"  n=64  unguided {sel_row.unguided_tanimoto:.4f} -> guided {sel_row.guided_tanimoto:.4f} "
          f"(gain {sel_row.tanimoto_gain:+.4f}), PB {sel_row.unguided_pb_valid:.3f} -> "
          f"{sel_row.guided_pb_valid:.3f}")

RERUN = False: skipping the GPU rerun. Stored value of the same configuration:
  n=64  unguided 0.2072 -> guided 0.2272 (gain +0.0199), PB 0.875 -> 0.875


## Summary of the mismatches found here

| item | paper | records |
|---|---|---|
| Table 3 multiMFM Tanimoto | 0.227 | 0.2272 at n=64, seed 0, best of a 4-point grid; 0.2241 vs 0.2287 unguided at n=4,096 × 3 (paired gain -0.0046 ± 0.0008) |
| molecules evaluated | 4,096 | 64 for the printed numbers |
| Table 21 unsteered PB | 92.0% | no 4,096-molecule measurement exists; 87.5% (n=64) / 88.0% (n=4,096 × 3) in the same runs; 92.0% occurs in the n=100 training validation at epochs 26/28/37, 98.0% at the released epoch 38 |
| guide predictor | "off-the-shelf from TFG-Flow" | TFG-Flow's architecture and FP2 label, weights trained by the authors in four legs (`checkpoints/geom/structure_head/train_args.json`) |
| Table 20 | 18 settings listed | `max_jump` 0.15, `anneal_frac` 0.2, ESD clips 8/10 and `diag_weight` 2 are omitted; the student was distilled on 150k of ~1.17M molecules |

None of these is resolved here: they are recorded for the authors' decision
(`docs/REPRODUCIBILITY_PLAN.md` §7).